In [1]:
# plotting
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator
import cartopy.crs as ccrs               
import cartopy.feature as cfeature         
import cartopy.util as cutil
#import plotting_module

# calculations
import xarray as xr                        
import numpy as np 
import pandas as pd

# regridding
import xesmf as xe

In this notebook: Script for matching model & measurements, and converting VMR to DU for monthly range. Here, end result includes DataArray representing 240 months of model output (January 2005-December 2024)

In [2]:
ozone_dataset = xr.open_dataset("/glade/campaign/acom/acom-climate/UTLS/shawnh/archive/FCnudged_f09.mam.mar27.2000_2021.002/atm/proc/tseries/month_1/FCnudged_f09.mam.mar27.2000_2021.002.cam.h0.O3.200201-202412.nc")
ps_dataset = xr.open_dataset('/glade/campaign/acom/acom-climate/UTLS/shawnh/archive/FCnudged_f09.mam.mar27.2000_2021.002/atm/proc/tseries/month_1/FCnudged_f09.mam.mar27.2000_2021.002.cam.h0.PS.200201-202412.nc')

In [3]:
ozone = ozone_dataset["O3"]

In [4]:
p0 = ozone_dataset["P0"]
hyai = ozone_dataset["hyai"]
hybi = ozone_dataset["hybi"]
ps = ps_dataset['PS']
lev = ozone_dataset.coords['lev']
num_lev = lev.shape[0]

In [5]:
# convert to hPa from Pa
p0 = p0.copy() / 100
ps = ps.copy() / 100

In [6]:
## Pick desired date range
start_date = '2005-02-01'
end_date = '2025-01-01'

# Slice ozone an the available months in the date range based on calendar months for both O3 and PS variables
truncated_ozone = ozone.sel(time=slice(start_date, end_date))
truncated_ozone = truncated_ozone.transpose('lev','time','lat','lon')

truncated_ps = ps.sel(time=slice(start_date, end_date))
truncated_ps = truncated_ps.transpose('time','lat','lon')

#ozone_monthly_mean = truncated_ozone.groupby('time.month').mean('time')

Tropospheric column calculations

In [7]:
# threshold value in hPa
threshold = 100

# constants / conversion factor
NAv = 6.0221415e+23                       # molecules in mole
g = 9.81                                  # gravity
MWair = 28.94                             # g/mol
xp_const = (NAv * 10)/(MWair*g)           # scaling factor, pa to hPa and cm to m
DU_CONVERSION = 2.69 * 10**16

In [8]:
# Initialize pressure edge arrays
mod_press_edge_top = xr.zeros_like(truncated_ozone)
mod_press_edge_bottom = xr.zeros_like(truncated_ozone)

Calculating pressure at hybrid levels

p(k) = a(k) * p0 + b(k) * ps

In [9]:
# Calculate pressure edge arrays
# Indices start at the top and end at the bottom
for i in range(num_lev):
    mod_press_edge_top[i,:,:,:] = hyai[i]*p0 + hybi[i]*truncated_ps
    mod_press_edge_bottom[i,:,:,:] = hyai[i+1]*p0 + hybi[i+1]*truncated_ps

#print(mod_press_edge_bottom.sel(lat=slice(40,40.5),lon=slice(150,150.5), month=1).values)

In [10]:
filtered_300hpa_upper = mod_press_edge_top.where(mod_press_edge_top >= threshold, drop=False)
filtered_300hpa_lower = mod_press_edge_bottom.where(mod_press_edge_bottom >= threshold, drop=False)

In [11]:
filtered_deltap = filtered_300hpa_lower - filtered_300hpa_upper

pres_top_v_300 = mod_press_edge_top - threshold
pres_bottom_v_300 = mod_press_edge_bottom - threshold

In [12]:
filtered_deltap_sliver = pres_bottom_v_300

# Delta p sliver where sign changes for bottom/top interfaces
filtered_deltap_sliver = filtered_deltap_sliver.where(pres_bottom_v_300 >= 0)
filtered_deltap_sliver = filtered_deltap_sliver.where(pres_top_v_300 < 0)

In [13]:
filtered_deltap_sliver = filtered_deltap_sliver.fillna(0)

In [14]:
filtered_deltap = filtered_deltap.fillna(0)

In [15]:
# Sum up both the layer edge and the pressure column to the ground
combined_filtered_deltap = filtered_deltap + filtered_deltap_sliver

In [ ]:
ozone_array = xr.zeros_like(truncated_ozone)

In [ ]:
ozone_array = truncated_ozone.where(combined_filtered_deltap > 0)

In [ ]:
ozone_array = ozone_array.fillna(0)

In [ ]:
ozone_column = xr.dot(combined_filtered_deltap, xp_const*ozone_array, dims='lev')
ozone_full_du_column = ozone_column.copy() / DU_CONVERSION

In [ ]:
ozone_full_du_column.mean(dim={'yearmonth','lat','lon'}, skipna=True)

In [ ]:
ozone_full_du_column

In [ ]:
# Make into netCDF for easy access to model output

# Rename to match observations
#ozone_full_du_column = ozone_full_du_column.rename({'time':'yearmonth'})
#ozone_full_du_column

# Change to proper date range
#times = pd.date_range('2005-01-01', periods=240, freq='MS')
#ozone_full_du_column = ozone_full_du_column.assign_coords(yearmonth=times)

# Change name to proper threshold value, in this case ground-to-100hPa
#ozone_full_du_column.to_netcdf(path="/glade/u/home/mvoncyga/SOARS_2025/data/datasets_full/100hpa_tco_cesm_monthly.nc", format="NETCDF4")